# VSP5600 Performance Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/visubramaniam/PerformanceExportAnalysis/blob/main/VSP5600_Performance_Analysis.ipynb)

## 1. Configuration & Global Constants

**IMPORTANT:** Update these values when analyzing a different dataset.

In [ ]:
# =============================================================================
# GLOBAL CONSTANTS - UPDATE THESE FOR DIFFERENT DATASETS
# =============================================================================

from pathlib import Path
from datetime import datetime

# Root directory containing the performance data
ROOT_DIR = Path("/Users/visubramaniam/Downloads/PerformanceExportAnalysis")

# Name of the ZIP file to extract (without .zip extension)
DATA_FOLDER_NAME = "out_VSP5200_61794_Performance_Data.20251222.0922"

# Generate timestamp for output folder
EXTRACTION_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

# Output folder with timestamp suffix (where ZIP contents will be extracted)
OUT_FOLDER = ROOT_DIR / f"out_{EXTRACTION_TIMESTAMP}"

# BASE_PATH points to the out folder (same as OUT_FOLDER)
BASE_PATH = OUT_FOLDER

# Output folder for merged/processed data
MERGED_FOLDER = ROOT_DIR / "merged"

# Archive folder for processed zip files
ARCHIVE_FOLDER = ROOT_DIR / "archive"

# =============================================================================
# IMAGE/PDF EXPORT CONSTANTS
# =============================================================================
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
IMAGES_FOLDER = ROOT_DIR / "images"
PDF_OUTPUT_FILE = ROOT_DIR / f"VSP5600_Performance_Report_{TIMESTAMP}.pdf"
PLOT_DPI = 150
PLOT_FORMAT = 'png'

# =============================================================================
# CSV PARSING CONSTANTS
# =============================================================================
CSV_SKIP_ROWS = 6  # Number of metadata rows to skip in CSV files

# =============================================================================
# ANALYSIS THRESHOLDS (Initial values - will be adaptively adjusted)
# =============================================================================

# IOPS and Transfer Rate Thresholds
MBPS_THRESHOLD = 100          # MB/s threshold for transfer rate analysis
KBPS_THRESHOLD = 100000       # KB/s threshold (MBPS_THRESHOLD * 1000)
MIN_IOPS_THRESHOLD = 100      # Minimum IOPS to consider LDEV active
LDEV_TRANSRATE_THRESHOLD = KBPS_THRESHOLD  # Transfer rate threshold

# Response Time Thresholds (microseconds)
PORT_RESPONSE_THRESHOLD = 5000    # Port response time threshold (µs)
LDEV_RESPONSE_THRESHOLD = 1000    # LDEV response time threshold (µs)
HIGH_RESPONSE_THRESHOLD = 5000    # High response threshold for filtering (µs)
MIN_RESPONSE_THRESHOLD = 2000     # Minimum response threshold (µs)

# Consecutive Reading Thresholds
MIN_CONSECUTIVE = 5           # Minimum consecutive readings above threshold

# Read Analysis Thresholds
MIN_READ_PCT = 60             # Minimum read percentage to consider read-heavy

# =============================================================================
# ADAPTIVE THRESHOLD SETTINGS
# =============================================================================
MIN_ITEMS_THRESHOLD = 5       # Minimum number of LDEVs/ports to find
GT_ADJUST_FACTOR = 0.80       # Multiply by 0.80 (decrease by 20%) for > thresholds
LT_ADJUST_FACTOR = 1.10       # Multiply by 1.10 (increase by 10%) for < thresholds
MAX_ITERATIONS = 20           # Maximum iterations to prevent infinite loops

# =============================================================================
# DATA SUBFOLDER PATHS (relative to BASE_PATH)
# =============================================================================
PHY_MPU_FOLDER = "PhyMPU_dat"
PHY_PROC_FOLDER = "PhyProc_dat"
PORT_FOLDER = "Port_dat"
LDEV_FOLDER = "LDEVEachOfCU_dat"

print(f"ROOT_DIR: {ROOT_DIR}")
print(f"OUT_FOLDER: {OUT_FOLDER}")
print(f"BASE_PATH: {BASE_PATH}")
print(f"Data folder exists: {BASE_PATH.exists()}")
print(f"Images folder: {IMAGES_FOLDER}")

## 2. Import Libraries

In [ ]:
# Standard library imports
import glob
import shutil
import zipfile
from pathlib import Path

# Data processing
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
import seaborn as sns

# PDF generation
from PIL import Image

# Set plot style
sns.set_style("whitegrid")

# Initialize list to track saved plot files
saved_plots = []

# Create images folder if it doesn't exist
IMAGES_FOLDER.mkdir(exist_ok=True)

print("All libraries imported successfully!")
print(f"Images will be saved to: {IMAGES_FOLDER}")

## 3. Function Definitions

### 3.1 File Operations

In [ ]:
# =============================================================================
# FILE OPERATIONS - ZIP EXTRACTION
# =============================================================================

def find_all_zip_files(root_path, recursive=False):
    """
    Find all zip files in the root path directory (case-insensitive).
    
    Args:
        root_path: Path object or string to the root directory
        recursive: If True, search recursively in all subdirectories
        
    Returns:
        List of Path objects for each zip file found
    """
    root = Path(root_path)
    if recursive:
        # Search recursively using ** pattern
        zip_files = list(root.glob("**/*.zip")) + list(root.glob("**/*.ZIP"))
    else:
        # Search only in the immediate directory
        zip_files = list(root.glob("*.zip")) + list(root.glob("*.ZIP"))
    return zip_files


def unzip_file(zip_path, extract_to=None):
    """
    Extract a zip file to the specified directory.
    Handles the case where the zip contains a single root folder with the same name
    as the zip file to avoid nested folders (e.g., PhyMPU_dat/PhyMPU_dat/).
    
    Args:
        zip_path: Path to the zip file
        extract_to: Directory to extract files to (default: same folder as zip)
        
    Returns:
        Path to the extracted folder
    """
    zip_path = Path(zip_path)
    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        # Get all entries in the zip
        namelist = zip_ref.namelist()
        
        # Check if zip has a single root folder matching the zip filename
        root_folders = set()
        for name in namelist:
            # Get the first component of the path
            parts = name.split('/')
            if parts[0]:
                root_folders.add(parts[0])
        
        # If there's exactly one root folder and it matches the zip stem, extract to parent
        single_root = len(root_folders) == 1
        root_name = list(root_folders)[0] if single_root else None
        zip_has_matching_root = single_root and root_name == zip_path.stem
        
        if extract_to is None:
            if zip_has_matching_root:
                # Extract directly to parent folder (zip contents will create the folder)
                extract_to = zip_path.parent
            else:
                # Extract to a subfolder named after the zip file (without extension)
                extract_to = zip_path.parent / zip_path.stem
        
        extract_to = Path(extract_to)
        extract_to.mkdir(parents=True, exist_ok=True)
        
        zip_ref.extractall(extract_to)
    
    # Return the actual extracted folder path
    if zip_has_matching_root:
        return extract_to / root_name
    return extract_to


def archive_zip_file(zip_path, archive_folder):
    """
    Move a processed zip file to the archive folder.
    
    Args:
        zip_path: Path to the zip file
        archive_folder: Path to the archive directory
    """
    archive = Path(archive_folder)
    archive.mkdir(parents=True, exist_ok=True)
    
    dest = archive / Path(zip_path).name
    
    # Check if destination already exists
    if dest.exists():
        # Add a counter to make unique filename
        counter = 1
        while dest.exists():
            dest = archive / f"{Path(zip_path).stem}_{counter}{Path(zip_path).suffix}"
            counter += 1
    
    shutil.move(str(zip_path), str(dest))
    print(f"  Archived: {Path(zip_path).name} -> {dest}")


def extract_all_nested_zips(base_path):
    """
    Recursively find and extract all ZIP files in a directory tree.
    Extracted ZIP files are deleted (not archived) after extraction.
    
    Args:
        base_path: Path to search for ZIP files
        
    Returns:
        Number of ZIP files extracted
    """
    total_extracted = 0
    iteration = 0
    max_iterations = 10  # Prevent infinite loops
    
    while iteration < max_iterations:
        iteration += 1
        zip_files = find_all_zip_files(base_path, recursive=True)
        
        if not zip_files:
            break
            
        print(f"\nIteration {iteration}: Found {len(zip_files)} ZIP file(s)")
        
        for zip_path in zip_files:
            print(f"  Extracting: {zip_path.relative_to(base_path)}")
            try:
                extract_path = unzip_file(zip_path)
                print(f"    -> {extract_path.relative_to(base_path)}")
                # Delete the extracted ZIP file (nested ZIPs are not archived)
                zip_path.unlink()
                total_extracted += 1
            except Exception as e:
                print(f"    ⚠️  Error: {e}")
    
    return total_extracted


print("File operation functions defined (with recursive ZIP extraction).")

### 3.2 CSV Data Loading Functions

In [ ]:
def strip_ldev_suffix(ldev_name):
    """
    Remove the trailing 'X' suffix from LDEV names (indicates thin volume).
    E.g., '00:00:EAX' becomes '00:00:EA'
    
    Args:
        ldev_name: LDEV name string
        
    Returns:
        LDEV name without the X suffix
    """
    if isinstance(ldev_name, str) and ldev_name.endswith('X'):
        return ldev_name[:-1]
    return ldev_name


def load_csv_with_skip(file_path, skip_rows=None):
    """
    Load a single CSV file with skip rows.
    
    Args:
        file_path: Path to the CSV file
        skip_rows: Number of rows to skip (defaults to CSV_SKIP_ROWS)
        
    Returns:
        pandas DataFrame
    """
    if skip_rows is None:
        skip_rows = CSV_SKIP_ROWS
    df = pd.read_csv(file_path, skiprows=skip_rows)
    # Remove any duplicate header rows (where 'time' column contains literal 'time')
    df = df[df['time'] != 'time'].copy()
    df['time'] = pd.to_datetime(df['time'])
    return df


def load_and_combine_csv(folder_path, file_pattern, skip_rows=None):
    """
    Load all CSV files matching pattern from a folder and combine them.
    Each file may have different columns - they are merged horizontally on time.
    
    Args:
        folder_path: Path to the folder containing CSV files
        file_pattern: Glob pattern to match files (e.g., "LDEV_*.csv")
        skip_rows: Number of rows to skip (defaults to CSV_SKIP_ROWS)
        
    Returns:
        Combined pandas DataFrame, or empty DataFrame if no files found
    """
    if skip_rows is None:
        skip_rows = CSV_SKIP_ROWS
        
    files = sorted(glob.glob(str(Path(folder_path) / file_pattern)))
    print(f"Found {len(files)} files: {[Path(f).name for f in files]}")
    
    # Return empty DataFrame if no files found
    if len(files) == 0:
        print("  ⚠️  No files found matching pattern")
        return pd.DataFrame(columns=['time'])
    
    # Load first file as base
    combined = pd.read_csv(files[0], skiprows=skip_rows)
    combined = combined[combined['time'] != 'time'].copy()
    combined['time'] = pd.to_datetime(combined['time'])
    
    # Convert data columns to numeric
    data_cols = [col for col in combined.columns if col not in ['No.', 'time']]
    for col in data_cols:
        combined[col] = pd.to_numeric(combined[col], errors='coerce')
    
    # Merge remaining files (they have different LDEV columns)
    for f in files[1:]:
        df = pd.read_csv(f, skiprows=skip_rows)
        df = df[df['time'] != 'time'].copy()
        df['time'] = pd.to_datetime(df['time'])
        
        # Convert data columns to numeric
        df_data_cols = [col for col in df.columns if col not in ['No.', 'time']]
        for col in df_data_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # Drop No. column if exists for merge
        if 'No.' in df.columns:
            df = df.drop(columns=['No.'])
        
        # Merge on time to add new columns
        combined = combined.merge(df, on='time', how='outer')
    
    # Drop No. column if exists in combined
    if 'No.' in combined.columns:
        combined = combined.drop(columns=['No.'])
    
    combined = combined.sort_values('time').reset_index(drop=True)
    
    return combined


def load_iops_data(file_path, ldev_cols, metric_suffix, skip_rows=None):
    """
    Load IOPS data and melt to long format.
    
    Args:
        file_path: Path to the CSV file
        ldev_cols: List of LDEV column names
        metric_suffix: Suffix to add to metric name (e.g., '_read_iops')
        skip_rows: Number of rows to skip
        
    Returns:
        DataFrame in long format with columns: time, LDEV, metric
    """
    if skip_rows is None:
        skip_rows = CSV_SKIP_ROWS
        
    df = pd.read_csv(file_path, skiprows=skip_rows)
    df['time'] = pd.to_datetime(df['time'])
    
    # Melt to long format
    df_melted = df.melt(
        id_vars=['time'],
        value_vars=ldev_cols,
        var_name='LDEV',
        value_name=metric_suffix.strip('_')
    )
    
    return df_melted

### 3.3 Data Filtering Functions

In [ ]:
# =============================================================================
# DATA FILTERING FUNCTIONS
# =============================================================================

def has_consecutive_high_values(series, threshold, min_consecutive):
    """
    Check if series has at least min_consecutive values above threshold.
    
    Args:
        series: pandas Series of numeric values
        threshold: Value threshold to check against
        min_consecutive: Minimum number of consecutive values required
        
    Returns:
        Boolean indicating if condition is met
    """
    above_threshold = (series > threshold).astype(int)
    consecutive_count = 0
    max_consecutive = 0
    
    for val in above_threshold:
        if val == 1:
            consecutive_count += 1
            max_consecutive = max(max_consecutive, consecutive_count)
        else:
            consecutive_count = 0
            
    return max_consecutive >= min_consecutive


def has_consecutive_above_threshold(series, threshold, min_consecutive):
    """
    Check if series has at least min_consecutive values >= threshold.
    Uses rolling window for efficiency.
    
    Args:
        series: pandas Series of numeric values
        threshold: Value threshold to check against
        min_consecutive: Minimum number of consecutive values required
        
    Returns:
        Boolean indicating if condition is met
    """
    above_threshold = (series >= threshold).astype(int)
    consecutive_count = above_threshold.rolling(
        window=min_consecutive, 
        min_periods=min_consecutive
    ).sum()
    return (consecutive_count >= min_consecutive).any()


def filter_active_rows(df, data_cols):
    """
    Keep rows where at least one data column has positive value (activity).
    
    Args:
        df: DataFrame to filter
        data_cols: List of data column names to check
        
    Returns:
        Filtered DataFrame
    """
    mask = (df[data_cols] > 0).any(axis=1)
    return df[mask].copy()


def get_active_columns(df, data_cols):
    """
    Return columns that have at least some positive values.
    
    Args:
        df: DataFrame to check
        data_cols: List of column names to evaluate
        
    Returns:
        List of column names with positive values
    """
    active_cols = []
    for col in data_cols:
        if (df[col] > 0).any():
            active_cols.append(col)
    return active_cols


def filter_columns_by_positive_values(df, data_cols):
    """
    Filter DataFrame to keep only columns with at least one positive value.
    
    Args:
        df: DataFrame to filter
        data_cols: List of data column names to check
        
    Returns:
        tuple: (filtered DataFrame, list of kept columns, list of dropped columns)
    """
    cols_with_positive = ['No.', 'time'] if 'No.' in df.columns else ['time']
    
    for col in data_cols:
        if col in df.columns and (df[col] >= 0).any():
            cols_with_positive.append(col)
    
    cols_dropped = [col for col in data_cols if col not in cols_with_positive]
    df_filtered = df[cols_with_positive].copy()
    
    return df_filtered, cols_with_positive, cols_dropped


def adaptive_threshold_filter_gt(df, columns, initial_threshold, min_consecutive, 
                                  min_items=5, adjust_factor=0.80, max_iter=20,
                                  filter_func=None):
    """
    Adaptively filter columns using > threshold, decreasing threshold until min_items found.
    
    Args:
        df: DataFrame containing the data
        columns: List of column names to filter
        initial_threshold: Starting threshold value
        min_consecutive: Minimum consecutive readings above threshold
        min_items: Minimum number of items to find (default 5)
        adjust_factor: Factor to multiply threshold by each iteration (default 0.80 = 20% decrease)
        max_iter: Maximum iterations to prevent infinite loops
        filter_func: Function to use for filtering (default: has_consecutive_high_values)
        
    Returns:
        tuple: (list of filtered columns, final threshold used)
    """
    if filter_func is None:
        filter_func = has_consecutive_high_values
    
    threshold = initial_threshold
    filtered_items = []
    
    for iteration in range(max_iter):
        filtered_items = []
        for col in columns:
            if col in df.columns:
                # Skip columns with all NaN
                if df[col].isna().all():
                    continue
                if filter_func(df[col].fillna(-999), threshold, min_consecutive):
                    filtered_items.append(col)
        
        if len(filtered_items) >= min_items:
            print(f"  ✓ Found {len(filtered_items)} items at threshold {threshold:.2f} (iteration {iteration + 1})")
            return filtered_items, threshold
        
        # Decrease threshold for > comparisons
        new_threshold = threshold * adjust_factor
        if iteration < max_iter - 1:
            print(f"  → Adjusting threshold: {threshold:.2f} -> {new_threshold:.2f} (found {len(filtered_items)}, need {min_items})")
        threshold = new_threshold
    
    # Return whatever we found after max iterations
    print(f"  ⚠️  Max iterations reached. Found {len(filtered_items)} items at threshold {threshold:.2f}")
    return filtered_items, threshold


def adaptive_threshold_filter_lt(df, columns, initial_threshold, min_consecutive,
                                  min_items=5, adjust_factor=1.10, max_iter=20,
                                  filter_func=None):
    """
    Adaptively filter columns using < threshold, increasing threshold until min_items found.
    
    Args:
        df: DataFrame containing the data
        columns: List of column names to filter
        initial_threshold: Starting threshold value  
        min_consecutive: Minimum consecutive readings below threshold
        min_items: Minimum number of items to find (default 5)
        adjust_factor: Factor to multiply threshold by each iteration (default 1.10 = 10% increase)
        max_iter: Maximum iterations to prevent infinite loops
        filter_func: Custom filter function (receives series, threshold, min_consecutive)
        
    Returns:
        tuple: (list of filtered columns, final threshold used)
    """
    threshold = initial_threshold
    filtered_items = []
    
    def default_lt_filter(series, thresh, min_consec):
        """Check if series has consecutive values below threshold."""
        below_threshold = (series < thresh).astype(int)
        consecutive_count = 0
        max_consecutive = 0
        for val in below_threshold:
            if val == 1:
                consecutive_count += 1
                max_consecutive = max(max_consecutive, consecutive_count)
            else:
                consecutive_count = 0
        return max_consecutive >= min_consec
    
    if filter_func is None:
        filter_func = default_lt_filter
    
    for iteration in range(max_iter):
        filtered_items = []
        for col in columns:
            if col in df.columns:
                if df[col].isna().all():
                    continue
                if filter_func(df[col].fillna(999999), threshold, min_consecutive):
                    filtered_items.append(col)
        
        if len(filtered_items) >= min_items:
            print(f"  ✓ Found {len(filtered_items)} items at threshold {threshold:.2f} (iteration {iteration + 1})")
            return filtered_items, threshold
        
        # Increase threshold for < comparisons
        new_threshold = threshold * adjust_factor
        if iteration < max_iter - 1:
            print(f"  → Adjusting threshold: {threshold:.2f} -> {new_threshold:.2f} (found {len(filtered_items)}, need {min_items})")
        threshold = new_threshold
    
    print(f"  ⚠️  Max iterations reached. Found {len(filtered_items)} items at threshold {threshold:.2f}")
    return filtered_items, threshold


def adaptive_filter_by_value(df, columns, initial_threshold, comparison='gt',
                             min_items=5, adjust_factor=None, max_iter=20):
    """
    Adaptively filter columns where ANY value meets threshold criteria.
    
    Args:
        df: DataFrame containing the data
        columns: List of column names to filter
        initial_threshold: Starting threshold value
        comparison: 'gt' for greater than, 'lt' for less than, 'gte' for >=, 'lte' for <=
        min_items: Minimum number of items to find
        adjust_factor: Factor to adjust threshold (default: 0.80 for gt/gte, 1.10 for lt/lte)
        max_iter: Maximum iterations
        
    Returns:
        tuple: (list of filtered columns, final threshold used)
    """
    if adjust_factor is None:
        adjust_factor = 0.80 if comparison in ['gt', 'gte'] else 1.10
    
    threshold = initial_threshold
    
    for iteration in range(max_iter):
        filtered_items = []
        for col in columns:
            if col in df.columns:
                if df[col].isna().all():
                    continue
                if comparison == 'gt':
                    if (df[col] > threshold).any():
                        filtered_items.append(col)
                elif comparison == 'gte':
                    if (df[col] >= threshold).any():
                        filtered_items.append(col)
                elif comparison == 'lt':
                    if (df[col] < threshold).any():
                        filtered_items.append(col)
                elif comparison == 'lte':
                    if (df[col] <= threshold).any():
                        filtered_items.append(col)
        
        if len(filtered_items) >= min_items:
            print(f"  ✓ Found {len(filtered_items)} items at threshold {threshold:.2f} (iteration {iteration + 1})")
            return filtered_items, threshold
        
        new_threshold = threshold * adjust_factor
        if iteration < max_iter - 1:
            print(f"  → Adjusting threshold: {threshold:.2f} -> {new_threshold:.2f} (found {len(filtered_items)}, need {min_items})")
        threshold = new_threshold
    
    print(f"  ⚠️  Max iterations reached. Found {len(filtered_items)} items at threshold {threshold:.2f}")
    return filtered_items, threshold


print("Data filtering functions defined (including adaptive threshold filtering).")

## 4. Data Extraction

Extract zip files and prepare data folders.

In [ ]:
# Look for the main data ZIP file in ROOT_DIR
# Expected: out_VSP5600_XXXXX_Performance_Data.YYYYMMDD.HHMM.zip
main_zip_file = ROOT_DIR / f"{DATA_FOLDER_NAME}.zip"

print(f"Looking for main ZIP file: {main_zip_file.name}")
print(f"  Path: {main_zip_file}")
print(f"  Exists: {main_zip_file.exists()}")

if main_zip_file.exists():
    print(f"\n✅ Found main ZIP file ({main_zip_file.stat().st_size / (1024*1024):.1f} MB)")
    print(f"   Will extract to: {OUT_FOLDER}")
elif BASE_PATH.exists():
    print(f"\n✅ Data folder already exists: {BASE_PATH}")
    print("   (Main ZIP may have been extracted previously)")
else:
    print(f"\n⚠️  Neither main ZIP file nor data folder found!")

In [ ]:
# Extract main ZIP file to out folder, then recursively extract all nested ZIPs

if main_zip_file.exists():
    # Create the output folder
    OUT_FOLDER.mkdir(parents=True, exist_ok=True)
    print(f"Created output folder: {OUT_FOLDER}")
    
    # Step 1: Extract the main ZIP file to the output folder
    print(f"\nStep 1: Extracting main ZIP file: {main_zip_file.name}")
    try:
        extract_path = unzip_file(main_zip_file, OUT_FOLDER)
        print(f"  -> Extracted to: {extract_path}")
        
        # Archive the main ZIP file
        archive_zip_file(main_zip_file, ARCHIVE_FOLDER)
    except Exception as e:
        print(f"  ⚠️  Error extracting main ZIP: {e}")
    
    # Step 2: Recursively extract all nested ZIPs in the data folder
    print("\n" + "="*60)
    print("Step 2: Extracting nested ZIP files in data folder...")
    print("="*60)
    total = extract_all_nested_zips(BASE_PATH)
    print(f"\nTotal nested ZIPs extracted: {total}")
    
elif BASE_PATH.exists():
    # Data folder exists, check for any remaining nested ZIPs
    print("Main ZIP already extracted. Checking for nested ZIPs...")
    total = extract_all_nested_zips(BASE_PATH)
    if total > 0:
        print(f"\nExtracted {total} nested ZIP file(s)")
    else:
        print("No nested ZIP files found.")
else:
    print("⚠️  No ZIP file or data folder found. Cannot proceed.")

print("\nExtraction complete!")

In [ ]:
# Verify data folder structure
print(f"Checking data folder: {BASE_PATH}")
print(f"Exists: {BASE_PATH.exists()}")

if BASE_PATH.exists():
    print("\nSubfolders found:")
    for item in sorted(BASE_PATH.iterdir()):
        if item.is_dir():
            print(f"  📁 {item.name}")
            
    # Create merged folder for output
    MERGED_FOLDER.mkdir(exist_ok=True)
    print(f"\nMerged output folder: {MERGED_FOLDER}")

## 5. Data Loading & Processing

### 5.1 Cache Metrics (Write Pending Rate & Cache Usage Rate)

In [ ]:
# Load Cache Metrics data
cache_path = BASE_PATH / PHY_MPU_FOLDER

if not cache_path.exists():
    print(f"⚠️  Cache folder not found: {cache_path}")
    print("   Skipping Cache Metrics analysis...")
    df_cache_metrics = pd.DataFrame(columns=['time'])
    df_write_pending = pd.DataFrame(columns=['time', 'Write_Pending_Rate'])
    df_cache_usage = pd.DataFrame(columns=['time', 'Cache_Usage_Rate'])
else:
    # Load Write Pending Rate
    write_pending_file = cache_path / "PHY_Short_Write_Pending_Rate.csv"
    if write_pending_file.exists():
        df_write_pending = load_csv_with_skip(write_pending_file)
        # Find the appropriate column for Write Pending Rate (try common names)
        write_pending_col = None
        for col_name in ['Total(CLPR00_DKC00)', 'ALL', 'Total']:
            if col_name in df_write_pending.columns:
                write_pending_col = col_name
                break
        if write_pending_col:
            df_write_pending['Write_Pending_Rate'] = df_write_pending[write_pending_col]
            print(f"Write Pending Rate: {len(df_write_pending)} rows (using column: {write_pending_col})")
        else:
            # Use first numeric column after 'time'
            numeric_cols = [c for c in df_write_pending.columns if c not in ['No.', 'time']]
            if numeric_cols:
                write_pending_col = numeric_cols[0]
                df_write_pending['Write_Pending_Rate'] = df_write_pending[write_pending_col]
                print(f"Write Pending Rate: {len(df_write_pending)} rows (using column: {write_pending_col})")
            else:
                print(f"⚠️  No suitable Write Pending Rate column found. Available: {df_write_pending.columns.tolist()}")
                df_write_pending['Write_Pending_Rate'] = 0
    else:
        print(f"⚠️  Write Pending Rate file not found: {write_pending_file}")
        df_write_pending = pd.DataFrame(columns=['time', 'Write_Pending_Rate'])

    # Load Cache Usage Rate
    cache_usage_file = cache_path / "PHY_Short_Cache_Usage_Rate.csv"
    if cache_usage_file.exists():
        df_cache_usage = load_csv_with_skip(cache_usage_file)
        # Find the appropriate column for Cache Usage Rate (try common names)
        cache_usage_col = None
        for col_name in ['Total(CLPR00_DKC00)', 'ALL', 'Total']:
            if col_name in df_cache_usage.columns:
                cache_usage_col = col_name
                break
        if cache_usage_col:
            df_cache_usage['Cache_Usage_Rate'] = df_cache_usage[cache_usage_col]
            print(f"Cache Usage Rate: {len(df_cache_usage)} rows (using column: {cache_usage_col})")
        else:
            # Use first numeric column after 'time'
            numeric_cols = [c for c in df_cache_usage.columns if c not in ['No.', 'time']]
            if numeric_cols:
                cache_usage_col = numeric_cols[0]
                df_cache_usage['Cache_Usage_Rate'] = df_cache_usage[cache_usage_col]
                print(f"Cache Usage Rate: {len(df_cache_usage)} rows (using column: {cache_usage_col})")
            else:
                print(f"⚠️  No suitable Cache Usage Rate column found. Available: {df_cache_usage.columns.tolist()}")
                df_cache_usage['Cache_Usage_Rate'] = 0
    else:
        print(f"⚠️  Cache Usage Rate file not found: {cache_usage_file}")
        df_cache_usage = pd.DataFrame(columns=['time', 'Cache_Usage_Rate'])

    # Merge cache metrics
    if df_write_pending.empty or df_cache_usage.empty or 'Write_Pending_Rate' not in df_write_pending.columns:
        df_cache_metrics = pd.DataFrame(columns=['time'])
    else:
        df_cache_metrics = df_write_pending[['time', 'Write_Pending_Rate']].merge(
            df_cache_usage[['time', 'Cache_Usage_Rate']],
            on='time',
            how='outer'
        ).sort_values('time')
        print(f"Merged Cache Metrics: {len(df_cache_metrics)} rows")

### 5.2 MPU Usage Data

In [ ]:
# Load MPU Usage data from PhyProc folder
proc_path = BASE_PATH / PHY_PROC_FOLDER
mpu_file = proc_path / "PHY_Short_MP.csv"

if not proc_path.exists():
    print(f"⚠️  Processor folder not found: {proc_path}")
    print("   Skipping MPU Usage analysis...")
    df_mpu_raw = pd.DataFrame(columns=['time'])
    mpu_cols = []
elif not mpu_file.exists():
    print(f"⚠️  MPU file not found: {mpu_file}")
    print("   Skipping MPU Usage analysis...")
    df_mpu_raw = pd.DataFrame(columns=['time'])
    mpu_cols = []
else:
    df_mpu_raw = load_csv_with_skip(mpu_file)
    print(f"MPU Raw Data: {df_mpu_raw.shape}")

    # Get MPU columns (excluding No. and time)
    mpu_cols = [col for col in df_mpu_raw.columns if col not in ['No.', 'time']]
    print(f"MPU columns: {len(mpu_cols)}")

In [ ]:
# FIX: Redefine get_mpu_id function to properly extract MPU IDs
# Column format: 'MPU-010.MP010-00' -> extract 'MPU-010'

if not mpu_cols:
    print("⚠️  MPU data not available - skipping MPU grouping")
    mpu_groups = {}
else:
    print("Sample mpu_cols:", mpu_cols[:3])

    def get_mpu_id(col_name):
        """Extract MPU ID from column name like 'MPU-010.MP010-00' -> 'MPU-010'"""
        if '.' in col_name:
            return col_name.split('.')[0]
        return None

    # Recreate MPU grouping with corrected function
    mpu_groups = {}
    for col in mpu_cols:
        mpu_id = get_mpu_id(col)
        if mpu_id:
            if mpu_id not in mpu_groups:
                mpu_groups[mpu_id] = []
            mpu_groups[mpu_id].append(col)

    print(f"\nMPU groups: {len(mpu_groups)}")
    for mpu_id, cols in sorted(mpu_groups.items()):
        print(f"  {mpu_id}: {len(cols)} cores")

In [ ]:
# Calculate average MPU usage for each MPU group
if not mpu_groups or df_mpu_raw.empty:
    print("⚠️  MPU data not available - skipping MPU average calculation")
    df_mpu_avg = pd.DataFrame(columns=['time'])
    df_mpu_merged = pd.DataFrame(columns=['time'])
    mpu_avg_cols = []
else:
    df_mpu_avg = df_mpu_raw[['time']].copy()

    for mpu_id, cols in sorted(mpu_groups.items()):
        # Filter to only numeric columns that exist in the dataframe
        valid_cols = [c for c in cols if c in df_mpu_raw.columns]
        if valid_cols:
            # Select only these columns and convert to numeric, coercing errors
            numeric_data = df_mpu_raw[valid_cols].apply(pd.to_numeric, errors='coerce')
            df_mpu_avg[f'{mpu_id}_avg'] = numeric_data.mean(axis=1)
        else:
            print(f"⚠️  No valid columns found for {mpu_id}")

    print(f"MPU averages shape: {df_mpu_avg.shape}")
    print(f"Columns: {df_mpu_avg.columns.tolist()}")

    # Get MPU average columns
    mpu_avg_cols = [col for col in df_mpu_avg.columns if col.endswith('_avg')]

    # Merge MPU with Write Pending Rate if available
    if 'Write_Pending_Rate' in df_write_pending.columns and not df_write_pending.empty:
        df_mpu_merged = df_mpu_avg.merge(
            df_write_pending[['time', 'Write_Pending_Rate']],
            on='time',
            how='inner'
        )
        print(f"Merged MPU with Write Pending shape: {df_mpu_merged.shape}")
    else:
        df_mpu_merged = df_mpu_avg.copy()
        print("Write Pending Rate not available, using MPU averages only")

### 5.3 HIE Metrics (HIE ISW & MPU HIE)

In [ ]:
# Load HIE Metrics
hie_path = BASE_PATH / PHY_MPU_FOLDER

if not hie_path.exists():
    print(f"⚠️  HIE folder not found: {hie_path}")
    print("   Skipping HIE Metrics analysis...")
    df_hie_isw = pd.DataFrame(columns=['time'])
    df_hie_usage = pd.DataFrame(columns=['time'])
else:
    # Load HIE_ISW
    hie_isw_file = hie_path / "PHY_Short_HIE_ISW.csv"
    if hie_isw_file.exists():
        df_hie_isw = load_csv_with_skip(hie_isw_file)
        print(f"HIE_ISW: {df_hie_isw.shape}")
    else:
        print(f"⚠️  HIE_ISW file not found: {hie_isw_file}")
        df_hie_isw = pd.DataFrame(columns=['time'])

    # Load MPU HIE (HIE Usage per MPU)
    hie_usage_file = hie_path / "PHY_Short_MPU_HIE.csv"
    if hie_usage_file.exists():
        df_hie_usage = load_csv_with_skip(hie_usage_file)
        print(f"MPU_HIE: {df_hie_usage.shape}")
    else:
        print(f"⚠️  MPU_HIE file not found: {hie_usage_file}")
        df_hie_usage = pd.DataFrame(columns=['time'])

### 5.4 Port Response Data

In [ ]:
# Load Port Response data
port_path = BASE_PATH / PORT_FOLDER
port_file = port_path / "Port_Response.csv"

if not port_path.exists():
    print(f"⚠️  Port folder not found: {port_path}")
    print("   Skipping Port Response analysis...")
    df_port_response = pd.DataFrame(columns=['time'])
elif not port_file.exists():
    print(f"⚠️  Port Response file not found: {port_file}")
    print("   Skipping Port Response analysis...")
    df_port_response = pd.DataFrame(columns=['time'])
else:
    df_port_response = load_csv_with_skip(port_file)
    print(f"Port Response: {df_port_response.shape}")

    # Clean negative values (set to NaN)
    port_cols = [col for col in df_port_response.columns if col not in ['No.', 'time']]
    for col in port_cols:
        df_port_response.loc[df_port_response[col] < 0, col] = np.nan
    
    print(f"Port columns: {len(port_cols)}")

In [ ]:
# Filter ports with high response times (using adaptive threshold)
if df_port_response.empty or len(df_port_response.columns) <= 1:
    print("⚠️  Port Response data not available - skipping analysis")
    high_response_ports = []
    df_port_filtered = pd.DataFrame(columns=['time'])
    port_threshold_used = PORT_RESPONSE_THRESHOLD
else:
    port_cols = [col for col in df_port_response.columns if col not in ['No.', 'time']]
    
    print(f"Filtering ports with consecutive readings > threshold (starting at {PORT_RESPONSE_THRESHOLD} µs)...")
    high_response_ports, port_threshold_used = adaptive_threshold_filter_gt(
        df_port_response, 
        port_cols, 
        PORT_RESPONSE_THRESHOLD, 
        MIN_CONSECUTIVE,
        min_items=MIN_ITEMS_THRESHOLD,
        adjust_factor=GT_ADJUST_FACTOR,
        max_iter=MAX_ITERATIONS
    )
    
    print(f"Ports with {MIN_CONSECUTIVE}+ consecutive readings > {port_threshold_used:.0f} µs: {len(high_response_ports)}")

    # Keep filtered port data
    if high_response_ports:
        df_port_filtered = df_port_response[['time'] + high_response_ports].copy()
        print(f"Filtered port data shape: {df_port_filtered.shape}")
    else:
        df_port_filtered = pd.DataFrame(columns=['time'])
        print("No ports found meeting criteria after adaptive adjustment")

### 5.5 LDEV IOPS Data

In [ ]:
# Load LDEV IOPS data (Read and Write)
ldev_path = BASE_PATH / LDEV_FOLDER

if not ldev_path.exists():
    print(f"⚠️  LDEV folder not found: {ldev_path}")
    print("   Skipping LDEV IOPS analysis...")
    df_read_iops = pd.DataFrame(columns=['time'])
    df_write_iops = pd.DataFrame(columns=['time'])
else:
    print("Loading Read IOPS...")
    read_iops_path = ldev_path / "LDEV_Read_IOPS"
    if read_iops_path.exists():
        df_read_iops = load_and_combine_csv(read_iops_path, "LDEV_Read_IOPS*.csv")
        print(f"  Shape: {df_read_iops.shape}")
    else:
        print(f"  ⚠️  Read IOPS folder not found: {read_iops_path}")
        df_read_iops = pd.DataFrame(columns=['time'])

    print("\nLoading Write IOPS...")
    write_iops_path = ldev_path / "LDEV_Write_IOPS"
    if write_iops_path.exists():
        df_write_iops = load_and_combine_csv(write_iops_path, "LDEV_Write_IOPS*.csv")
        print(f"  Shape: {df_write_iops.shape}")
    else:
        print(f"  ⚠️  Write IOPS folder not found: {write_iops_path}")
        df_write_iops = pd.DataFrame(columns=['time'])

In [ ]:
# Find common LDEV columns between Read and Write IOPS
if df_read_iops.empty or df_write_iops.empty or len(df_read_iops.columns) <= 1:
    print("⚠️  LDEV IOPS data not available - skipping LDEV analysis")
    ldev_cols_read = []
    ldev_cols_write = []
    common_ldevs = []
    df_read_iops_filtered = pd.DataFrame(columns=['time'])
    df_write_iops_filtered = pd.DataFrame(columns=['time'])
    active_read_cols = []
    active_write_cols = []
    active_iops_ldevs = []
else:
    ldev_cols_read = [col for col in df_read_iops.columns if col not in ['No.', 'time']]
    ldev_cols_write = [col for col in df_write_iops.columns if col not in ['No.', 'time']]
    common_ldevs = list(set(ldev_cols_read) & set(ldev_cols_write))
    print(f"Common LDEV columns: {len(common_ldevs)}")

    # Filter active rows
    df_read_iops_filtered = filter_active_rows(df_read_iops, common_ldevs)
    df_write_iops_filtered = filter_active_rows(df_write_iops, common_ldevs)
    print(f"Read IOPS after filtering: {len(df_read_iops_filtered)} rows, unique times: {df_read_iops_filtered['time'].nunique()}")
    print(f"Write IOPS after filtering: {len(df_write_iops_filtered)} rows, unique times: {df_write_iops_filtered['time'].nunique()}")
    
    # CRITICAL FIX: Deduplicate on 'time' column to ensure one row per timestamp
    # This handles cases where multiple CSV files or data points map to the same timestamp
    df_read_iops_filtered = df_read_iops_filtered.drop_duplicates(subset=['time'], keep='first').reset_index(drop=True)
    df_write_iops_filtered = df_write_iops_filtered.drop_duplicates(subset=['time'], keep='first').reset_index(drop=True)
    print(f"After deduplication - Read: {len(df_read_iops_filtered)} rows, Write: {len(df_write_iops_filtered)} rows")

    # Get active columns
    active_read_cols = get_active_columns(df_read_iops_filtered, common_ldevs)
    active_write_cols = get_active_columns(df_write_iops_filtered, common_ldevs)
    active_iops_ldevs = list(set(active_read_cols) & set(active_write_cols))
    print(f"LDEVs with activity in both Read and Write: {len(active_iops_ldevs)}")


### 5.6 LDEV Transfer Rate Data

In [ ]:
# Load LDEV Transfer Rate data (all CU files - hex suffixes 00-FF)
if not ldev_path.exists() or df_read_iops.empty:
    print("⚠️  LDEV data not available - skipping Transfer Rate analysis")
    df_transrate_combined = pd.DataFrame(columns=['time'])
else:
    transrate_path = ldev_path / "LDEV_TransRate"
    if transrate_path.exists():
        print("Loading Transfer Rate (all CUs)...")
        df_transrate_combined = load_and_combine_csv(transrate_path, "LDEV_TransRate*.csv")
        print(f"  Combined Shape: {df_transrate_combined.shape}")
        if not df_transrate_combined.empty:
            print(f"  Columns: {len(df_transrate_combined.columns) - 1} LDEVs")
    else:
        print(f"  ⚠️  Transfer Rate folder not found: {transrate_path}")
        df_transrate_combined = pd.DataFrame(columns=['time'])

# Backward compatibility
df_transrate00 = df_transrate_combined
df_transrate01 = df_transrate_combined

In [ ]:
# Filter LDEV Transfer Rate for high transfer rate LDEVs (using adaptive threshold)
if 'df_transrate_combined' not in dir() or df_transrate_combined.empty or len(df_transrate_combined.columns) <= 1:
    print("⚠️  Transfer Rate data not available - skipping analysis")
    high_transrate_ldevs = []
    transrate_threshold_used = LDEV_TRANSRATE_THRESHOLD
else:
    transrate_cols = [col for col in df_transrate_combined.columns if col not in ['No.', 'time']]
    
    print(f"Filtering LDEVs with high transfer rates (starting at {LDEV_TRANSRATE_THRESHOLD} KB/s)...")
    
    # Find LDEVs with high transfer rates using adaptive threshold
    high_transrate_ldevs, transrate_threshold_used = adaptive_threshold_filter_gt(
        df_transrate_combined,
        transrate_cols,
        LDEV_TRANSRATE_THRESHOLD,
        MIN_CONSECUTIVE,
        min_items=MIN_ITEMS_THRESHOLD,
        adjust_factor=GT_ADJUST_FACTOR,
        max_iter=MAX_ITERATIONS
    )

    print(f"High transfer rate LDEVs: {len(high_transrate_ldevs)} (threshold: {transrate_threshold_used:.0f} KB/s)")
    if high_transrate_ldevs:
        print(f"  Example LDEVs: {high_transrate_ldevs[:10]}...")

# Backward compatibility
high_transrate_ldevs_00 = high_transrate_ldevs if 'high_transrate_ldevs' in dir() else []
high_transrate_ldevs_01 = high_transrate_ldevs if 'high_transrate_ldevs' in dir() else []

### 5.7 LDEV Response Data

In [ ]:
# Load LDEV Response data (all files with hex suffixes 00-FF)
if not ldev_path.exists() or df_read_iops.empty:
    print("⚠️  LDEV data not available - skipping Response analysis")
    df_response_all = pd.DataFrame(columns=['time'])
else:
    response_path = ldev_path / "LDEV_Response"
    print("Loading all LDEV Response files (hex suffixes 00-FF)...")
    if response_path.exists():
        # Load all LDEV_Response*.csv files (covers 00-FF hex suffixes)
        df_response_all = load_and_combine_csv(response_path, "LDEV_Response*.csv")
        print(f"  Combined Shape: {df_response_all.shape}")
        if not df_response_all.empty:
            print(f"  Columns: {len(df_response_all.columns) - 1} LDEVs")  # minus time column
    else:
        print(f"  ⚠️  Response folder not found: {response_path}")
        df_response_all = pd.DataFrame(columns=['time'])

In [ ]:
# Filter LDEV Response for high response time LDEVs (using adaptive threshold)
if 'df_response_all' not in dir() or df_response_all.empty or len(df_response_all.columns) <= 1:
    print("⚠️  Response data not available - skipping analysis")
    high_response_ldevs = []
    ldev_response_threshold_used = LDEV_RESPONSE_THRESHOLD
else:
    response_cols = [col for col in df_response_all.columns if col not in ['No.', 'time']]

    print(f"Filtering LDEVs with consecutive readings > threshold (starting at {LDEV_RESPONSE_THRESHOLD} µs)...")
    high_response_ldevs, ldev_response_threshold_used = adaptive_threshold_filter_gt(
        df_response_all,
        response_cols,
        LDEV_RESPONSE_THRESHOLD,
        MIN_CONSECUTIVE,
        min_items=MIN_ITEMS_THRESHOLD,
        adjust_factor=GT_ADJUST_FACTOR,
        max_iter=MAX_ITERATIONS
    )

    print(f"High response LDEVs found: {len(high_response_ldevs)} (threshold: {ldev_response_threshold_used:.0f} µs)")
    if high_response_ldevs:
        print(f"  Example LDEVs: {high_response_ldevs[:10]}...")

In [ ]:
# DIAGNOSTIC: Check shapes before merge
print(f"df_read_iops_filtered shape: {df_read_iops_filtered.shape}")
print(f"df_write_iops_filtered shape: {df_write_iops_filtered.shape}")
print(f"df_read_iops_filtered['time'] nunique: {df_read_iops_filtered['time'].nunique()}")
print(f"df_write_iops_filtered['time'] nunique: {df_write_iops_filtered['time'].nunique()}")
print(f"Active IOPS LDEVs count: {len(active_iops_ldevs)}")

In [ ]:
# Quick diagnostic to check input data shapes
print(f'df_read_iops_filtered shape: {df_read_iops_filtered.shape}')
print(f'df_write_iops_filtered shape: {df_write_iops_filtered.shape}')
print(f'df_read_iops_filtered time unique: {df_read_iops_filtered["time"].nunique()}')
print(f'df_write_iops_filtered time unique: {df_write_iops_filtered["time"].nunique()}')
print(f'active_iops_ldevs count: {len(active_iops_ldevs)}')
print(f'\nCheck for duplicate times in read filtered:')
print(f'  Total rows: {len(df_read_iops_filtered)}')
print(f'  Unique times: {df_read_iops_filtered["time"].nunique()}')
print(f'  Avg rows per time: {len(df_read_iops_filtered) / df_read_iops_filtered["time"].nunique():.2f}')

In [ ]:
# Calculate Total IOPS (Read + Write) for Read Hit analysis
if df_read_iops.empty or df_write_iops.empty or not active_iops_ldevs:
    print("⚠️  LDEV IOPS data not available - skipping Read Hit analysis")
    df_iops_merged = pd.DataFrame(columns=['time'])
    df_total_iops = pd.DataFrame(columns=['time'])
else:
    # Since we've already deduplicated at source, just select the relevant columns and merge
    df_read_subset = df_read_iops_filtered[['time'] + active_iops_ldevs].copy()
    df_write_subset = df_write_iops_filtered[['time'] + active_iops_ldevs].copy()
    
    # Rename columns for clarity
    df_read_subset = df_read_subset.rename(columns={col: f"{col}_read" for col in active_iops_ldevs})
    df_write_subset = df_write_subset.rename(columns={col: f"{col}_write" for col in active_iops_ldevs})
    
    # Merge on time
    df_iops_merged = df_read_subset.merge(df_write_subset, on='time', how='inner')
    
    # Fill any NaN values with 0
    for ldev in active_iops_ldevs:
        df_iops_merged[f"{ldev}_read"] = df_iops_merged[f"{ldev}_read"].fillna(0)
        df_iops_merged[f"{ldev}_write"] = df_iops_merged[f"{ldev}_write"].fillna(0)

    # Calculate total IOPS using efficient dictionary-based approach
    data_dict = {'time': df_iops_merged['time'].values}
    for ldev in active_iops_ldevs:
        read_vals = df_iops_merged[f"{ldev}_read"].clip(lower=0)
        write_vals = df_iops_merged[f"{ldev}_write"].clip(lower=0)
        data_dict[ldev] = (read_vals + write_vals).values
    
    df_total_iops = pd.DataFrame(data_dict)

    print(f"Total IOPS shape: {df_total_iops.shape}")
    print(f"Unique timestamps in merged data: {df_iops_merged['time'].nunique()}")


In [ ]:
# Filter LDEVs with total IOPS >= threshold (using adaptive threshold)
if df_total_iops.empty or len(df_total_iops.columns) <= 1 or not active_iops_ldevs:
    print("⚠️  Total IOPS data not available - skipping read percentage analysis")
    high_iops_ldevs = []
    df_read_pct = pd.DataFrame(columns=['time'])
    read_heavy_ldevs = []
    iops_threshold_used = MIN_IOPS_THRESHOLD
else:
    print(f"Filtering LDEVs with IOPS >= threshold (starting at {MIN_IOPS_THRESHOLD})...")
    high_iops_ldevs, iops_threshold_used = adaptive_filter_by_value(
        df_total_iops,
        active_iops_ldevs,
        MIN_IOPS_THRESHOLD,
        comparison='gte',
        min_items=MIN_ITEMS_THRESHOLD,
        adjust_factor=GT_ADJUST_FACTOR,  # decrease threshold to find more
        max_iter=MAX_ITERATIONS
    )
    print(f"LDEVs with IOPS >= {iops_threshold_used:.0f}: {len(high_iops_ldevs)}")

    # Calculate Read Percentage
    df_read_pct = df_total_iops[['time']].copy()
    for ldev in high_iops_ldevs:
        read_vals = df_iops_merged[f"{ldev}_read"].clip(lower=0)
        total_vals = df_total_iops[ldev]
        df_read_pct[ldev] = np.where(total_vals > 0, (read_vals / total_vals) * 100, 0)

    # Filter read-heavy LDEVs (read pct >= MIN_READ_PCT at least once)
    # Use adaptive filtering for MIN_READ_PCT (less than comparison - lower read% is also interesting)
    print(f"Filtering read-heavy LDEVs (>= {MIN_READ_PCT}% read)...")
    read_heavy_ldevs, read_pct_threshold_used = adaptive_filter_by_value(
        df_read_pct,
        high_iops_ldevs,
        MIN_READ_PCT,
        comparison='gte',
        min_items=MIN_ITEMS_THRESHOLD,
        adjust_factor=GT_ADJUST_FACTOR,  # decrease % threshold to find more
        max_iter=MAX_ITERATIONS
    )
    print(f"Read-heavy LDEVs (>= {read_pct_threshold_used:.0f}% read): {len(read_heavy_ldevs)}")

In [ ]:
# Summary of IOPS filtering (already computed above)
print(f"Summary of IOPS-based filtering:")
print(f"  - Active IOPS LDEVs: {len(active_iops_ldevs) if 'active_iops_ldevs' in dir() else 0}")
print(f"  - High IOPS LDEVs: {len(high_iops_ldevs) if 'high_iops_ldevs' in dir() else 0}")
print(f"  - Read-heavy LDEVs: {len(read_heavy_ldevs) if 'read_heavy_ldevs' in dir() else 0}")

In [ ]:
# Load Read Response and Read Hits for read-heavy LDEVs
if not ldev_path.exists() or not read_heavy_ldevs:
    print("⚠️  LDEV data or read-heavy LDEVs not available - skipping Read Response/Hits analysis")
    df_read_resp = pd.DataFrame(columns=['time'])
    df_read_hits = pd.DataFrame(columns=['time'])
    common_read_analysis_ldevs = []
else:
    print("Loading Read Response...")
    read_resp_path = ldev_path / "LDEV_Read_Response"
    if read_resp_path.exists():
        df_read_resp = load_and_combine_csv(read_resp_path, "LDEV_Read_Response*.csv")
        print(f"  Shape: {df_read_resp.shape}")
    else:
        print(f"  ⚠️  Read Response folder not found")
        df_read_resp = pd.DataFrame(columns=['time'])

    print("\nLoading Read Hits...")
    read_hits_path = ldev_path / "LDEV_Read_Hit"
    if read_hits_path.exists():
        df_read_hits = load_and_combine_csv(read_hits_path, "LDEV_Read_Hit*.csv")
        print(f"  Shape: {df_read_hits.shape}")
    else:
        print(f"  ⚠️  Read Hits folder not found")
        df_read_hits = pd.DataFrame(columns=['time'])

    # Find common LDEVs with all metrics
    if df_read_resp.empty or df_read_hits.empty:
        common_read_analysis_ldevs = []
    else:
        resp_cols = [col for col in df_read_resp.columns if col not in ['No.', 'time']]
        hits_cols = [col for col in df_read_hits.columns if col not in ['No.', 'time']]
        common_read_analysis_ldevs = list(
            set(read_heavy_ldevs) & set(resp_cols) & set(hits_cols)
        )
    print(f"\nLDEVs with complete read analysis data: {len(common_read_analysis_ldevs)}")

---

## 6. Visualizations

All charts and graphs are collected in this section.

### 6.1 Cache Metrics: Write Pending Rate vs Cache Usage Rate

In [ ]:
# Skip if required data is not available
if 'df_cache_metrics' not in dir() or df_cache_metrics is None or df_cache_metrics.empty:
    print("⚠️  Cache Metrics - Required data not available, skipping plot")
else:
    # Plot Cache Metrics
    fig, ax1 = plt.subplots(figsize=(14, 6))
    
    # Ensure time column is datetime
    if not pd.api.types.is_datetime64_any_dtype(df_cache_metrics['time']):
        df_cache_metrics['time'] = pd.to_datetime(df_cache_metrics['time'])
    
    # Extract date for subtitle
    date_str = df_cache_metrics['time'].iloc[0].strftime('%Y/%m/%d')
    
    # Plot Write Pending Rate on left axis
    ax1.plot(df_cache_metrics['time'], df_cache_metrics['Write_Pending_Rate'], 
             color='#1f77b4', linewidth=0.5, label='Write Pending Rate')
    ax1.set_xlabel('Time', fontsize=12)
    ax1.set_ylabel('Write Pending Rate (%)', color='#1f77b4', fontsize=12)
    ax1.tick_params(axis='y', labelcolor='#1f77b4')
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax1.tick_params(axis='x', rotation=45)
    
    # Plot Cache Usage Rate on right axis
    ax2 = ax1.twinx()
    ax2.plot(df_cache_metrics['time'], df_cache_metrics['Cache_Usage_Rate'],
             color='#2ca02c', linewidth=0.5, label='Cache Usage Rate')
    ax2.set_ylabel('Cache Usage Rate (%)', color='#2ca02c', fontsize=12)
    ax2.tick_params(axis='y', labelcolor='#2ca02c')
    
    plt.suptitle('Write Pending Rate vs Cache Usage Rate', fontsize=14, fontweight='bold')
    plt.title(f'Date: {date_str}', fontsize=10, style='italic')
    
    # Legend
    legend_elements = [
        Line2D([0], [0], color='#1f77b4', linewidth=1, label='Write Pending Rate (%)'),
        Line2D([0], [0], color='#2ca02c', linewidth=1, label='Cache Usage Rate (%)')
    ]
    ax1.legend(handles=legend_elements, loc='upper right', fontsize=9)
    
    plt.tight_layout()
    
    # Save plot
    plot_file = IMAGES_FOLDER / '01_cache_metrics.png'
    plt.savefig(plot_file, dpi=PLOT_DPI, bbox_inches='tight', facecolor='white')
    saved_plots.append(plot_file)
    print(f"Saved: {plot_file}")
    
    plt.show()

#### 6.1.1 Cache Metrics Analysis

#### .1  Analysis: Cache write pending rate is within normal limits.

### 6.2 MPU Usage vs Write Pending Rate

In [ ]:
# Skip if required data is not available
if 'df_mpu_merged' not in dir() or df_mpu_merged is None or df_mpu_merged.empty:
    print("⚠️  MPU Usage - Required data not available, skipping plot")
else:
    # Plot MPU Usage vs Write Pending Rate
    fig, ax1 = plt.subplots(figsize=(16, 6.5))
    
    # Ensure time column is datetime
    if not pd.api.types.is_datetime64_any_dtype(df_mpu_merged['time']):
        df_mpu_merged['time'] = pd.to_datetime(df_mpu_merged['time'])
    
    
    date_str = df_mpu_merged['time'].iloc[0].strftime('%Y/%m/%d')
    
    # MPU columns (excluding time and Write_Pending_Rate)
    mpu_avg_cols = sorted([col for col in df_mpu_merged.columns if col.endswith('_avg')])
    
    # Assign distinct colors for each MPU
    mpu_color_map = {
        'MPU-010': '#1f77b4',  # Blue
        'MPU-020': '#ff7f0e',  # Orange
        'MPU-110': '#2ca02c',  # Green
        'MPU-120': '#d62728',  # Red
    }
    default_colors = ['#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
    
    # Plot MPU Usage on left axis
    legend_elements = []
    for i, col in enumerate(mpu_avg_cols):
        # Get the MPU name (remove '_avg' suffix for display)
        mpu_name = col.replace('_avg', '')
        color = mpu_color_map.get(mpu_name, default_colors[i % len(default_colors)])
        
        ax1.plot(df_mpu_merged['time'], df_mpu_merged[col],
                 color=color, linewidth=0.8, label=mpu_name)
        legend_elements.append(Line2D([0], [0], color=color, linewidth=2, label=mpu_name))
    
    ax1.set_xlabel('Time', fontsize=12)
    ax1.set_ylabel('MPU Usage (%)', color='blue', fontsize=12)
    ax1.tick_params(axis='y', labelcolor='blue')
    # Set dynamic Y-axis for MPU usage based on data
    mpu_usage_max = max(df_mpu_merged[col].max() for col in mpu_avg_cols)
    ax1.set_ylim(0, min(mpu_usage_max * 1.1, 100))  # Cap at 100%  # Add 10% padding
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax1.tick_params(axis='x', rotation=45)
    
    # Plot Write Pending Rate on right axis
    ax2 = ax1.twinx()
    ax2.plot(df_mpu_merged['time'], df_mpu_merged['Write_Pending_Rate'],
             color='red', linewidth=0.8, linestyle='--', label='Write Pending Rate')
    ax2.set_ylabel('Write Pending Rate (%)', color='red', fontsize=12)
    ax2.tick_params(axis='y', labelcolor='red')
    # Set dynamic Y-axis for Write Pending Rate based on data
    write_pending_max = df_mpu_merged['Write_Pending_Rate'].max()
    ax2.set_ylim(0, min(write_pending_max * 1.1, 100))  # Cap at 100%  # Add 10% padding
    
    plt.suptitle('MPU Usage vs Write Pending Rate', fontsize=14, fontweight='bold')
    plt.title(f'Date: {date_str}', fontsize=10, style='italic')
    
    # Legend - add Write Pending Rate
    legend_elements.append(Line2D([0], [0], color='red', linewidth=2, linestyle='--', label='Write Pending Rate'))
    fig.subplots_adjust(right=0.82)
    ax1.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=10, framealpha=0.9)
    
    # Save plot
    plot_file = IMAGES_FOLDER / '02_mpu_usage.png'
    plt.savefig(plot_file, dpi=PLOT_DPI, bbox_inches='tight', facecolor='white')
    saved_plots.append(plot_file)
    print(f"Saved: {plot_file}")
    
    plt.show()

#### 6.2.1 MPU Usage Analysis

#### .1  Analysis: MPU usage is within normal operating range.

### 6.3 HIE Metrics: HIE ISW vs MPU HIE

In [ ]:
# Plot HIE Metrics
# First, create df_hie_metrics by merging HIE ISW and HIE Usage if not already defined
if 'df_hie_metrics' not in dir() or df_hie_metrics is None:
    # Check if source dataframes exist and are not empty
    if df_hie_isw.empty or df_hie_usage.empty:
        print("⚠️  HIE data not available - skipping HIE Metrics plot")
        df_hie_metrics = pd.DataFrame()
    else:
        # Identify the data column in each dataframe (excluding 'time' and 'No.')
        hie_isw_cols = [c for c in df_hie_isw.columns if c not in ['time', 'No.']]
        hie_usage_cols = [c for c in df_hie_usage.columns if c not in ['time', 'No.']]
        
        if hie_isw_cols and hie_usage_cols:
            # Rename the first data column to our standard names
            df_hie_isw_renamed = df_hie_isw[['time', hie_isw_cols[0]]].copy()
            df_hie_isw_renamed.columns = ['time', 'HIE_ISW']
            
            df_hie_usage_renamed = df_hie_usage[['time', hie_usage_cols[0]]].copy()
            df_hie_usage_renamed.columns = ['time', 'MPU_HIE']
            
            df_hie_metrics = df_hie_isw_renamed.merge(df_hie_usage_renamed, on='time', how='inner')
            print(f"Created df_hie_metrics: {df_hie_metrics.shape}")
        else:
            print("⚠️  Could not find data columns in HIE dataframes")
            df_hie_metrics = pd.DataFrame()

if df_hie_metrics.empty:
    print("⚠️  Skipping HIE Metrics plot - no data available")
else:
    fig, ax1 = plt.subplots(figsize=(14, 6))

    # Ensure time column is datetime
    if not pd.api.types.is_datetime64_any_dtype(df_hie_metrics['time']):
        df_hie_metrics['time'] = pd.to_datetime(df_hie_metrics['time'])

    date_str = df_hie_metrics['time'].iloc[0].strftime('%Y/%m/%d')

    # Plot HIE_ISW on left axis
    ax1.plot(df_hie_metrics['time'], df_hie_metrics['HIE_ISW'],
             color='#1f77b4', linewidth=0.5, label='HIE ISW')
    ax1.set_xlabel('Time', fontsize=12)
    ax1.set_ylabel('HIE ISW (%)', color='#1f77b4', fontsize=12)
    ax1.tick_params(axis='y', labelcolor='#1f77b4')
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax1.tick_params(axis='x', rotation=45)

    # Plot MPU_HIE on right axis
    ax2 = ax1.twinx()
    ax2.plot(df_hie_metrics['time'], df_hie_metrics['MPU_HIE'],
             color='#2ca02c', linewidth=0.5, label='MPU HIE')
    ax2.set_ylabel('MPU HIE (%)', color='#2ca02c', fontsize=12)
    ax2.tick_params(axis='y', labelcolor='#2ca02c')

    plt.suptitle('HIE ISW vs MPU HIE', fontsize=14, fontweight='bold')
    plt.title(f'Date: {date_str}', fontsize=10, style='italic')

    # Legend
    legend_elements = [
        Line2D([0], [0], color='#1f77b4', linewidth=1, label='HIE ISW (%)'),
        Line2D([0], [0], color='#2ca02c', linewidth=1, label='MPU HIE (%)')
    ]
    ax1.legend(handles=legend_elements, loc='upper right', fontsize=9)

    plt.tight_layout()

    # Save plot
    plot_file = IMAGES_FOLDER / '03_hie_metrics.png'
    plt.savefig(plot_file, dpi=PLOT_DPI, bbox_inches='tight', facecolor='white')
    saved_plots.append(plot_file)
    print(f"Saved: {plot_file}")

    plt.show()

#### 6.3.1 HIE Metrics Analysis

#### .1  Analysis: The chart does not show any saturation of the HIE - ISW and MPU - HIE internal paths. 

### 6.4 Port Response Time Analysis

In [ ]:
# Skip if required data is not available
if 'df_port_filtered' not in dir() or df_port_filtered is None or df_port_filtered.empty:
    print("⚠️  Port Response - Required data not available, skipping plot")
elif not high_response_ports:
    print(f"No ports with {MIN_CONSECUTIVE}+ consecutive readings > {PORT_RESPONSE_THRESHOLD} µs found")
else:
    # Plot Port Response Times (high response ports exist)
    # Merge port response with Write Pending Rate
    df_port_merged = df_port_filtered.merge(
        df_write_pending[['time', 'Write_Pending_Rate']],
        on='time',
        how='inner'
    )
    
    if df_port_merged.empty:
        print("⚠️  Port Response - No data after merging with Write Pending, skipping plot")
    else:
        fig, ax1 = plt.subplots(figsize=(16, 6.5))
        
        # Ensure time column is datetime
        if not pd.api.types.is_datetime64_any_dtype(df_port_merged['time']):
            df_port_merged['time'] = pd.to_datetime(df_port_merged['time'])
        
        date_str = df_port_merged['time'].iloc[0].strftime('%Y/%m/%d')
        
        # Plot Port Response on left axis
        port_colors = ['#1f77b4', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22']
        for i, col in enumerate(high_response_ports[:8]):  # Limit to 8 ports
            ax1.plot(df_port_merged['time'], df_port_merged[col] / 1000,  # Convert µs to ms
                     color=port_colors[i % len(port_colors)], linewidth=0.5, label=col)
        
        ax1.set_xlabel('Time', fontsize=12)
        ax1.set_ylabel('Port Response Time (ms)', color='blue', fontsize=12)
        ax1.tick_params(axis='y', labelcolor='blue')
        # Set dynamic Y-axis for Port Response based on data
        port_response_max = max(df_port_merged[col].max() for col in high_response_ports[:8]) / 1000
        ax1.set_ylim(0, min(port_response_max * 1.1, 20))  # Cap at 20ms  # Add 10% padding
        ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
        ax1.tick_params(axis='x', rotation=45)
        
        # Plot Write Pending Rate on right axis
        ax2 = ax1.twinx()
        ax2.plot(df_port_merged['time'], df_port_merged['Write_Pending_Rate'],
                 color='red', linewidth=0.8, linestyle='--', label='Write Pending Rate')
        ax2.set_ylabel('Write Pending Rate (%)', color='red', fontsize=12)
        ax2.tick_params(axis='y', labelcolor='red')
        # Set dynamic Y-axis for Write Pending Rate based on data
        write_pending_max = df_port_merged['Write_Pending_Rate'].max()
        ax2.set_ylim(0, min(write_pending_max * 1.1, 100))  # Cap at 100%  # Add 10% padding
        
        plt.suptitle('Port Response Time vs Write Pending Rate', fontsize=14, fontweight='bold')
        plt.title(f'Date: {date_str} | Threshold: {PORT_RESPONSE_THRESHOLD} µs', fontsize=10, style='italic')
        
        # Legend - build legend elements for all ports shown
        legend_elements = [Line2D([0], [0], color=port_colors[i % len(port_colors)], linewidth=2, label=col)
                           for i, col in enumerate(high_response_ports[:8])]
        legend_elements.append(Line2D([0], [0], color='red', linewidth=2, linestyle='--', label='Write Pending Rate (%)'))
        
        # Place legend outside plot on the right, with extra space
        fig.subplots_adjust(right=0.75)
        ax1.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=9, framealpha=0.9)
        
        # Save plot
        plot_file = IMAGES_FOLDER / '04_port_response.png'
        plt.savefig(plot_file, dpi=PLOT_DPI, bbox_inches='tight', facecolor='white')
        saved_plots.append(plot_file)
        print(f"Saved: {plot_file}")
        
        plt.show()

#### 6.4.1 Port Response Analysis

#### .1  Analysis: Port response times are in the normal operating range.

### 6.5 LDEV IOPS Analysis

In [ ]:
# Skip if required data is not available
if 'df_total_iops' not in dir() or df_total_iops is None or df_total_iops.empty:
    print("⚠️  LDEV IOPS - Required data not available, skipping plot")
else:
    # Plot LDEV IOPS vs Write Pending Rate (top 5 by average IOPS)
    # Calculate average total IOPS for each LDEV
    avg_iops = {ldev: df_total_iops[ldev].mean() for ldev in high_iops_ldevs}
    top_5_iops_ldevs = sorted(avg_iops.keys(), key=lambda x: avg_iops[x], reverse=True)[:5]
    
    print(f"Top 5 LDEVs by average IOPS:")
    for ldev in top_5_iops_ldevs:
        print(f"  {strip_ldev_suffix(ldev)}: {avg_iops[ldev]:.0f} IOPS")
    
    # Merge with Write Pending Rate
    df_iops_plot = df_total_iops[['time'] + top_5_iops_ldevs].merge(
        df_write_pending[['time', 'Write_Pending_Rate']],
        on='time',
        how='inner'
    )
    
    fig, ax1 = plt.subplots(figsize=(16, 8))
    
    # Ensure time column is datetime
    if not pd.api.types.is_datetime64_any_dtype(df_iops_plot['time']):
        df_iops_plot['time'] = pd.to_datetime(df_iops_plot['time'])
    
    date_str = df_iops_plot['time'].iloc[0].strftime('%Y/%m/%d')
    
    # Plot IOPS on left axis
    ldev_colors = ['#1f77b4', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
    for i, ldev in enumerate(top_5_iops_ldevs):
        ax1.plot(df_iops_plot['time'], df_iops_plot[ldev],
                 color=ldev_colors[i], linewidth=0.5, label=strip_ldev_suffix(ldev))
    
    ax1.set_xlabel('Time', fontsize=12)
    ax1.set_ylabel('IOPS', color='blue', fontsize=12)
    ax1.tick_params(axis='y', labelcolor='blue')
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax1.tick_params(axis='x', rotation=45)
    
    # Plot Write Pending Rate on right axis
    ax2 = ax1.twinx()
    ax2.plot(df_iops_plot['time'], df_iops_plot['Write_Pending_Rate'],
             color='red', linewidth=0.8, linestyle='--', label='Write Pending Rate')
    ax2.set_ylabel('Write Pending Rate (%)', color='red', fontsize=12)
    ax2.tick_params(axis='y', labelcolor='red')
    ax2.set_ylim(0, 100)
    
    plt.suptitle('LDEV IOPS vs Write Pending Rate (Top 5)', fontsize=14, fontweight='bold')
    plt.title(f'Date: {date_str}', fontsize=10, style='italic')
    
    # Legend
    legend_elements = [Line2D([0], [0], color=ldev_colors[i], linewidth=1, label=strip_ldev_suffix(ldev))
                       for i, ldev in enumerate(top_5_iops_ldevs)]
    legend_elements.append(Line2D([0], [0], color='red', linewidth=1, linestyle='--', label='Write Pending Rate (%)'))
    ax1.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.08, 1), fontsize=9)
    
    plt.tight_layout()
    
    # Save plot
    plot_file = IMAGES_FOLDER / '05_ldev_iops.png'
    plt.savefig(plot_file, dpi=PLOT_DPI, bbox_inches='tight', facecolor='white')
    saved_plots.append(plot_file)
    print(f"Saved: {plot_file}")
    
    plt.show()

#### 6.5.1 LDEV IOPS Analysis

#### .1  Analysis: The chart identifies the LDEVs with high IO activity.

### 6.6 LDEV Transfer Rate Analysis

In [ ]:
# Skip if required data is not available
if 'df_transrate00' not in dir() or df_transrate00 is None or df_transrate00.empty:
    print("⚠️  Transfer Rate - Required data not available, skipping plot")
elif not high_transrate_ldevs_00:
    print(f"No LDEVs with transfer rate >= {MBPS_THRESHOLD} MB/s found")
else:
    # Plot LDEV Transfer Rate vs Write Pending Rate
    # Get top 5 by average transfer rate
    avg_transrate = {ldev: df_transrate00[ldev].mean() for ldev in high_transrate_ldevs_00}
    top_5_transrate = sorted(avg_transrate.keys(), key=lambda x: avg_transrate[x], reverse=True)[:5]
    
    print(f"Top 5 LDEVs by average Transfer Rate:")
    for ldev in top_5_transrate:
        print(f"  {strip_ldev_suffix(ldev)}: {avg_transrate[ldev]/1000:.1f} MB/s")
    
    # Merge with Write Pending Rate
    df_trans_plot = df_transrate00[['time'] + top_5_transrate].merge(
        df_write_pending[['time', 'Write_Pending_Rate']],
        on='time',
        how='inner'
    )
    
    # Ensure time column is datetime
    if not pd.api.types.is_datetime64_any_dtype(df_trans_plot['time']):
        df_trans_plot['time'] = pd.to_datetime(df_trans_plot['time'])
    
    date_str = df_trans_plot['time'].iloc[0].strftime('%Y/%m/%d')
    
    fig, ax1 = plt.subplots(figsize=(16, 8))
    
    # Plot Transfer Rate on left axis (convert KB/s to MB/s)
    ldev_colors = ['#1f77b4', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
    for i, ldev in enumerate(top_5_transrate):
        ax1.plot(df_trans_plot['time'], df_trans_plot[ldev] / 1000,  # KB/s to MB/s
                 color=ldev_colors[i], linewidth=0.5, label=strip_ldev_suffix(ldev))
    
    ax1.set_xlabel('Time', fontsize=12)
    ax1.set_ylabel('Transfer Rate (MB/s)', color='blue', fontsize=12)
    ax1.tick_params(axis='y', labelcolor='blue')
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax1.tick_params(axis='x', rotation=45)
    
    # Plot Write Pending Rate on right axis
    ax2 = ax1.twinx()
    ax2.plot(df_trans_plot['time'], df_trans_plot['Write_Pending_Rate'],
             color='red', linewidth=0.8, linestyle='--', label='Write Pending Rate')
    ax2.set_ylabel('Write Pending Rate (%)', color='red', fontsize=12)
    ax2.tick_params(axis='y', labelcolor='red')
    ax2.set_ylim(0, 100)
    
    plt.suptitle('LDEV Transfer Rate vs Write Pending Rate (Top 5)', fontsize=14, fontweight='bold')
    plt.title(f'Date: {date_str} | Threshold: {MBPS_THRESHOLD} MB/s', fontsize=10, style='italic')
    
    # Legend
    legend_elements = [Line2D([0], [0], color=ldev_colors[i], linewidth=1, label=f'{strip_ldev_suffix(ldev)} (MB/s)')
                       for i, ldev in enumerate(top_5_transrate)]
    legend_elements.append(Line2D([0], [0], color='red', linewidth=1, linestyle='--', label='Write Pending Rate (%)'))
    ax1.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.08, 1), fontsize=9)
    
    plt.tight_layout()
    
    # Save plot
    plot_file = IMAGES_FOLDER / '06_ldev_transfer_rate.png'
    plt.savefig(plot_file, dpi=PLOT_DPI, bbox_inches='tight', facecolor='white')
    saved_plots.append(plot_file)
    print(f"Saved: {plot_file}")
    
    plt.show()

#### 6.6.1 Transfer Rate Analysis

#### .1  Analysis: The LDEVs that have high transfer rate are identified in this chart. 

### 6.7 LDEV Response Time Analysis

In [ ]:
# Skip if required data is not available
if 'df_response_all' not in dir() or df_response_all is None or df_response_all.empty:
    print("⚠️  LDEV Response - Required data not available, skipping plot")
elif 'high_response_ldevs' not in dir() or not high_response_ldevs:
    print(f"⚠️  No LDEVs with response time >= {LDEV_RESPONSE_THRESHOLD} µs found")
else:
    # Plot LDEV Response Time vs MPU Usage
    # Get top 5 by average response time
    avg_response = {}
    for ldev in high_response_ldevs:
        col_data = df_response_all[ldev].copy()
        col_data[col_data < 0] = np.nan
        avg_response[ldev] = col_data.mean()
    
    top_5_response = sorted(avg_response.keys(), key=lambda x: avg_response[x], reverse=True)[:5]
    
    print(f"Top 5 LDEVs by average Response Time:")
    for ldev in top_5_response:
        print(f"  {strip_ldev_suffix(ldev)}: {avg_response[ldev]:.0f} µs")
    
    # Merge with MPU data
    df_resp_plot = df_response_all[['time'] + top_5_response].merge(
        df_mpu_merged[['time'] + mpu_avg_cols],
        on='time',
        how='inner'
    )
    
    fig, ax1 = plt.subplots(figsize=(16, 8))
    
    # Ensure time column is datetime
    if not pd.api.types.is_datetime64_any_dtype(df_resp_plot['time']):
        df_resp_plot['time'] = pd.to_datetime(df_resp_plot['time'])
    
    date_str = df_resp_plot['time'].iloc[0].strftime('%Y/%m/%d')
    
    # Plot Response Time on left axis (convert µs to ms)
    ldev_colors = ['#1f77b4', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
    for i, ldev in enumerate(top_5_response):
        resp_data = df_resp_plot[ldev].copy()
        resp_data[resp_data < 0] = np.nan
        ax1.plot(df_resp_plot['time'], resp_data / 1000,  # µs to ms
                 color=ldev_colors[i], linewidth=0.5, label=strip_ldev_suffix(ldev))
    
    ax1.set_xlabel('Time', fontsize=12)
    ax1.set_ylabel('LDEV Response Time (ms)', color='blue', fontsize=12)
    ax1.tick_params(axis='y', labelcolor='blue')
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax1.tick_params(axis='x', rotation=45)
    
    # Plot MPU Usage on right axis
    ax2 = ax1.twinx()
    mpu_plot_cols = mpu_avg_cols[:4]  # Limit to 4 MPUs
    mpu_colors = ['#ff7f0e', '#17becf', '#bcbd22', '#e377c2']
    for i, col in enumerate(mpu_plot_cols):
        ax2.plot(df_resp_plot['time'], df_resp_plot[col],
                 color=mpu_colors[i], linewidth=0.5, linestyle='--', label=col)
    ax2.set_ylabel('MPU Usage (%)', color='green', fontsize=12)
    ax2.tick_params(axis='y', labelcolor='green')
    ax2.set_ylim(0, 100)
    
    plt.suptitle('LDEV Response Time vs MPU Usage (Top 5)', fontsize=14, fontweight='bold')
    plt.title(f'Date: {date_str}', fontsize=10, style='italic')
    
    # Legend
    legend_elements = [Line2D([0], [0], color=ldev_colors[i], linewidth=1, label=f'{strip_ldev_suffix(ldev)} (ms)')
                       for i, ldev in enumerate(top_5_response)]
    for i, col in enumerate(mpu_plot_cols):
        legend_elements.append(Line2D([0], [0], color=mpu_colors[i], linewidth=1, linestyle='--', label=f'{col} (%)'))
    ax1.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.12, 1), fontsize=8)
    
    plt.tight_layout()
    
    # Save plot
    plot_file = IMAGES_FOLDER / '07_ldev_response_time.png'
    plt.savefig(plot_file, dpi=PLOT_DPI, bbox_inches='tight', facecolor='white')
    saved_plots.append(plot_file)
    print(f"Saved: {plot_file}")
    
    plt.show()

#### 6.7.1 Response Time Analysis

#### .1  Analysis: LDEV response times during the monitoring period are shown.

### 6.8 LDEV Read Hit Analysis

In [ ]:
# Skip if required data is not available
if 'df_read_hits' not in dir() or df_read_hits is None or df_read_hits.empty:
    print("⚠️  Read Hit Analysis - Required data not available, skipping plot")
else:
    # Plot Read Hit Analysis for read-heavy LDEVs
    if common_read_analysis_ldevs:
        # Build final dataframe for read analysis
        df_read_pct_filtered = df_read_pct[['time'] + common_read_analysis_ldevs].copy()
        df_read_resp_clean = df_read_resp[['time'] + common_read_analysis_ldevs].copy()
        df_read_hits_clean = df_read_hits[['time'] + common_read_analysis_ldevs].copy()
        
        # Filter for high response LDEVs
        final_read_ldevs = []
        for ldev in common_read_analysis_ldevs:
            if has_consecutive_above_threshold(df_read_resp_clean[ldev], HIGH_RESPONSE_THRESHOLD, MIN_CONSECUTIVE):
                final_read_ldevs.append(ldev)
        
        print(f"LDEVs for Read Hit Analysis: {len(final_read_ldevs)}")
        
        # Plot each LDEV
        for idx, ldev in enumerate(final_read_ldevs[:5]):  # Limit to 5 LDEVs
            # Merge data for this LDEV
            df_ldev_plot = df_read_pct_filtered[['time', ldev]].rename(columns={ldev: 'read_pct'})
            df_ldev_plot = df_ldev_plot.merge(
                df_read_resp_clean[['time', ldev]].rename(columns={ldev: 'read_resp'}),
                on='time'
            )
            df_ldev_plot = df_ldev_plot.merge(
                df_read_hits_clean[['time', ldev]].rename(columns={ldev: 'read_hit'}),
                on='time'
            )
            
            fig, ax1 = plt.subplots(figsize=(14, 6))
            
            # Ensure time column is datetime
            if not pd.api.types.is_datetime64_any_dtype(df_ldev_plot['time']):
                df_ldev_plot['time'] = pd.to_datetime(df_ldev_plot['time'])
            
            date_str = df_ldev_plot['time'].iloc[0].strftime('%Y/%m/%d')
            
            # Left Y-axis: Read % and Read Hit %
            ax1.plot(df_ldev_plot['time'], df_ldev_plot['read_pct'], 
                     color='#9467bd', linewidth=0.5, label='Read %')
            ax1.plot(df_ldev_plot['time'], df_ldev_plot['read_hit'].clip(lower=0), 
                     color='#1f77b4', linewidth=0.5, label='Read Hit %')
            ax1.set_ylabel('Read % / Read Hit %', fontsize=11)
            ax1.set_ylim(0, 105)
            ax1.axhline(y=MIN_READ_PCT, color='red', linestyle=':', linewidth=1, alpha=0.7)
            ax1.set_xlabel('Time', fontsize=12)
            ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
            ax1.tick_params(axis='x', rotation=45)
            
            # Right Y-axis: Read Response Time
            ax2 = ax1.twinx()
            ax2.plot(df_ldev_plot['time'], df_ldev_plot['read_resp'].clip(lower=0),
                     color='#2ca02c', linewidth=0.5)
            ax2.set_ylabel('Read Response (µs)', color='#2ca02c', fontsize=12)
            ax2.tick_params(axis='y', labelcolor='#2ca02c')
            ax2.set_ylim(bottom=0)
            
            avg_pct = df_ldev_plot['read_pct'].mean()
            avg_hit = df_ldev_plot['read_hit'][df_ldev_plot['read_hit'] > 0].mean()
            
            plt.suptitle(f'LDEV {strip_ldev_suffix(ldev)} - Read Analysis', fontsize=14, fontweight='bold')
            plt.title(f'Date: {date_str} | Avg Read%: {avg_pct:.1f}% | Avg Read Hit: {avg_hit:.1f}%',
                      fontsize=10, style='italic')
            
            # Legend
            legend_elements = [
                Line2D([0], [0], color='#9467bd', linewidth=1, label='Read %'),
                Line2D([0], [0], color='#1f77b4', linewidth=1, label='Read Hit %'),
                Line2D([0], [0], color='red', linewidth=1, linestyle=':', label=f'{MIN_READ_PCT}% threshold'),
                Line2D([0], [0], color='#2ca02c', linewidth=1, label='Read Response (µs)')
            ]
            ax1.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.08, 1), fontsize=9)
            
            plt.tight_layout()
            
            # Save plot
            plot_file = IMAGES_FOLDER / f'08_ldev_read_hit_{idx+1:02d}_{ldev.replace(":", "_")}.png'
            plt.savefig(plot_file, dpi=PLOT_DPI, bbox_inches='tight', facecolor='white')
            saved_plots.append(plot_file)
            print(f"Saved: {plot_file}")
            
            plt.show()
            print("-" * 80)
    else:
        print("No LDEVs with complete read analysis data found")

#### 6.8.1 Read Hit Analysis

#### .1  Analysis: These set of charts shows the read hit%, read percentage and read response.

---

## 7. Export Results

Save processed data to CSV files and generate PDF report.

In [ ]:
# Ensure merged folder exists
MERGED_FOLDER.mkdir(exist_ok=True)

export_count = 0

# Export Cache Metrics
if 'df_cache_metrics' in dir() and df_cache_metrics is not None and not df_cache_metrics.empty:
    df_cache_metrics.to_csv(MERGED_FOLDER / 'cache_metrics.csv', index=False)
    print(f"  ✓ cache_metrics.csv ({len(df_cache_metrics)} rows)")
    export_count += 1
else:
    print("  ⚠️ cache_metrics.csv - skipped (no data)")

# Export HIE Metrics
if 'df_hie_metrics' in dir() and df_hie_metrics is not None and not df_hie_metrics.empty:
    df_hie_metrics.to_csv(MERGED_FOLDER / 'hie_metrics.csv', index=False)
    print(f"  ✓ hie_metrics.csv ({len(df_hie_metrics)} rows)")
    export_count += 1
else:
    print("  ⚠️ hie_metrics.csv - skipped (no data)")

# Export Port Response (filtered)
if 'high_response_ports' in dir() and high_response_ports:
    df_port_filtered.to_csv(MERGED_FOLDER / 'port_response_filtered.csv', index=False)
    print(f"  ✓ port_response_filtered.csv ({len(df_port_filtered)} rows, {len(high_response_ports)} ports)")
    export_count += 1
else:
    print("  ⚠️ port_response_filtered.csv - skipped (no high response ports)")

# Export Total IOPS
if 'high_iops_ldevs' in dir() and high_iops_ldevs and 'df_total_iops' in dir() and df_total_iops is not None and not df_total_iops.empty:
    df_total_iops[['time'] + high_iops_ldevs].to_csv(MERGED_FOLDER / 'ldev_total_iops.csv', index=False)
    print(f"  ✓ ldev_total_iops.csv ({len(df_total_iops)} rows, {len(high_iops_ldevs)} LDEVs)")
    export_count += 1
else:
    print("  ⚠️ ldev_total_iops.csv - skipped (no LDEV data)")

# Export Read Analysis data
if 'common_read_analysis_ldevs' in dir() and common_read_analysis_ldevs and 'df_read_pct' in dir() and df_read_pct is not None and not df_read_pct.empty:
    df_read_pct[['time'] + common_read_analysis_ldevs].to_csv(MERGED_FOLDER / 'ldev_read_pct.csv', index=False)
    print(f"  ✓ ldev_read_pct.csv ({len(df_read_pct)} rows)")
    export_count += 1
else:
    print("  ⚠️ ldev_read_pct.csv - skipped (no read analysis data)")

print(f"\nCSV exports complete! {export_count} files saved to: {MERGED_FOLDER}")

### 7.1 Generate PDF Report from Saved Plots

In [ ]:
# Generate Professional PDF Report with reportlab
import subprocess
import sys
import json

try:
    from reportlab.lib.pagesizes import A4, landscape, inch
    from reportlab.lib import colors
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, PageBreak
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_JUSTIFY
except ImportError:
    print("Installing reportlab...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "reportlab", "-q"])
    from reportlab.lib.pagesizes import A4, landscape, inch
    from reportlab.lib import colors
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, PageBreak
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_JUSTIFY

# Define Legal page size (8.5" x 14")
legal_width = 8.5 * inch
legal_height = 14 * inch
legal_page = (legal_width, legal_height)

from PIL import Image as PILImage

import json

def extract_analysis_from_notebook(notebook_path):
    """Extract analysis text from markdown cells in the notebook."""
    with open(notebook_path, 'r') as f:
        notebook = json.load(f)
    
    analysis_texts = {}
    current_chart_num = None
    
    for cell in notebook.get('cells', []):
        if cell.get('cell_type') == 'markdown':
            source = ''.join(cell.get('source', []))
            
            # Check for section headers like "### 6.1 Cache Metrics..."
            if source.startswith('###') and '6.' in source:
                try:
                    # Extract chart number from "### 6.X ..."
                    section_part = source.split('###')[1].strip()
                    if section_part.startswith('6.'):
                        chart_num_str = section_part.split()[0].replace('6.', '')
                        current_chart_num = int(chart_num_str)
                except (IndexError, ValueError):
                    pass
            
            # Check for analysis cells with format "#### .1  Analysis: <text>"
            if '####' in source and 'Analysis:' in source:
                if current_chart_num is not None:
                    # Extract text after "Analysis: "
                    parts = source.split('Analysis:', 1)
                    if len(parts) > 1:
                        analysis_text = parts[1].strip()
                        analysis_texts[current_chart_num] = analysis_text
    
    return analysis_texts

# Load analysis texts from notebook
notebook_path = Path.cwd() / 'VSP5600_Performance_Analysis.ipynb'
try:
    analysis_texts = extract_analysis_from_notebook(str(notebook_path))
except Exception as e:
    print(f"Warning: Could not load analysis texts from notebook: {e}")
    analysis_texts = {}

# Define the static chart files with titles
chart_titles = {
    '01_cache_metrics.png': '6.1 Cache Metrics: Write Pending Rate vs Cache Usage Rate',
    '02_mpu_usage.png': '6.2 MPU Usage Analysis',
    '03_hie_metrics.png': '6.3 HIE Metrics: Internal Path Utilization',
    '04_port_response.png': '6.4 Port Response Times',
    '05_ldev_iops.png': '6.5 LDEV IOPS Analysis',
    '06_ldev_transfer_rate.png': '6.6 LDEV Transfer Rate Analysis',
    '07_ldev_response_time.png': '6.7 LDEV Response Time vs MPU Usage'
}

# Build chart_data dictionary with programmatically extracted analysis
chart_data = {}
for idx, (chart_file, title) in enumerate(chart_titles.items(), 1):
    analysis = analysis_texts.get(idx, f'Analysis for chart {idx} not found in notebook.')
    chart_data[chart_file] = {
        'title': title,
        'analysis': analysis
    }

# Dynamically add all LDEV read hit charts
import glob
read_hit_charts = sorted(glob.glob(str(IMAGES_FOLDER / '08_ldev_read_hit_*.png')))
for idx, chart_path in enumerate(read_hit_charts, 1):
    chart_file = Path(chart_path).name
    chart_data[chart_file] = {
        'title': f'6.{7+idx} LDEV Read Hit Analysis - {chart_file.replace("08_ldev_read_hit_", "").replace(".png", "")}',
        'analysis': analysis_texts.get(8, 'These set of charts shows the read hit%, read percentage and read response.')
    }

print("Creating professional PDF report with Legal Landscape...")
print(f"Output file: {PDF_OUTPUT_FILE}")

# Create the PDF document with explicit margins to prevent image clipping
doc = SimpleDocTemplate(
    str(PDF_OUTPUT_FILE),
    pagesize=landscape(legal_page),
    leftMargin=0.3*inch,
    rightMargin=0.3*inch,
    topMargin=0.5*inch,
    bottomMargin=0.75*inch
)

# Define styles
styles = getSampleStyleSheet()

title_style = ParagraphStyle(
    'CustomTitle',
    parent=styles['Heading1'],
    fontSize=24,
    textColor=colors.HexColor('#1f4788'),
    spaceAfter=6,
    alignment=TA_CENTER,
    fontName='Helvetica-Bold'
)

subtitle_style = ParagraphStyle(
    'CustomSubtitle',
    parent=styles['Normal'],
    fontSize=14,
    textColor=colors.HexColor('#333333'),
    spaceAfter=12,
    alignment=TA_CENTER
)

chart_title_style = ParagraphStyle(
    'ChartTitle',
    parent=styles['Heading2'],
    fontSize=16,
    textColor=colors.HexColor('#1f4788'),
    spaceAfter=12,
    alignment=TA_LEFT,
    fontName='Helvetica-Bold'
)

analysis_style = ParagraphStyle(
    'AnalysisText',
    parent=styles['Normal'],
    fontSize=11,
    alignment=TA_JUSTIFY,
    spaceAfter=12,
    leading=14
)

elements = []

# ===== PAGE 1: TITLE PAGE =====
elements.append(Spacer(1, 2*inch))

# Title
elements.append(Paragraph("VSP5600 Performance Analysis Report", title_style))

# Subtitle
elements.append(Spacer(1, 0.3*inch))
elements.append(Paragraph("Storage Performance & Capacity Analysis", subtitle_style))

# Data details
elements.append(Spacer(1, 0.5*inch))
date_info = f"""
<font size=12><b>Report Generated:</b> {TIMESTAMP}<br/>
<b>Data Extraction Date:</b> {EXTRACTION_TIMESTAMP}</font>
"""
elements.append(Paragraph(date_info, styles['Normal']))

elements.append(Spacer(1, 0.8*inch))

# Footer text
footer_text = """
This report provides a comprehensive analysis of storage performance metrics 
collected from the Hitachi VSP5600 system. It includes detailed charts and 
analysis of cache performance, processor utilization, port response times, 
and LDEV (Logical Device) metrics.
"""
elements.append(Paragraph(footer_text, styles['Normal']))

# Add page break after title page
elements.append(PageBreak())

# ===== PAGES 2+: CHART + ANALYSIS PAGES =====
for chart_file, data in chart_data.items():
    chart_path = IMAGES_FOLDER / chart_file
    
    if not chart_path.exists():
        print(f"  ⚠️  Chart not found: {chart_file}")
        continue
    
    # Chart title
    elements.append(Paragraph(data['title'], chart_title_style))
    elements.append(Spacer(1, 0.2*inch))
    
    # Add chart image (resize to fit on page)
    try:
        img = PILImage.open(chart_path)
        img_width_px, img_height_px = img.size
        
        # Convert pixels to inches (matplotlib saves at 150 DPI)
        img_width_in = img_width_px / 150.0
        img_height_in = img_height_px / 150.0
        
        # Calculate scaling to achieve ~12" width while respecting max height
        min_width = 11.5  # inches
        max_height = 6.5  # inches
        
        scale_for_width = min_width / img_width_in if img_width_in > 0 else 1
        scale_for_height = max_height / img_height_in if img_height_in > 0 else 1
        scale_factor = min(scale_for_width, scale_for_height)
        
        if scale_factor * img_width_in < min_width:
            scale_factor = min_width / img_width_in
        
        final_width = scale_factor * img_width_in * inch
        final_height = scale_factor * img_height_in * inch
        
        img_element = RLImage(str(chart_path), width=final_width, height=final_height)
        elements.append(img_element)
        print(f"  ✓ Added: {chart_file}")
        
    except Exception as e:
        print(f"  ✗ Error adding image {chart_file}: {e}")
        continue
    
    elements.append(Spacer(1, 0.15*inch))
    
    # Add analysis text
    elements.append(Paragraph("<b>Analysis:</b>", styles['Normal']))
    elements.append(Paragraph(data['analysis'], analysis_style))
    
    # Add page break (except after last chart)
    if chart_file != list(chart_data.keys())[-1]:
        elements.append(PageBreak())

# Build the PDF
try:
    doc.build(elements)
    print(f"\n✅ PDF report created successfully!")
    print(f"   File: {PDF_OUTPUT_FILE}")
    total_pages = 1 + len(chart_data)  # 1 title page + analysis pages
    print(f"   Total Pages: {total_pages}")
    print(f"   Includes: 1 Title Page + {len(chart_data)} Analysis Pages with Charts & Text")
except Exception as e:
    print(f"❌ Error creating PDF: {e}")

---

## Summary

This notebook analyzed the following metrics from the Hitachi VSP5600 storage system:

1. **Cache Metrics** - Write Pending Rate vs Cache Usage Rate
2. **MPU Usage** - Processor utilization across MPU groups  
3. **HIE Metrics** - HIE ISW vs MPU HIE
4. **Port Response** - Port response times with high latency detection
5. **LDEV IOPS** - Read/Write IOPS for high-activity LDEVs
6. **LDEV Transfer Rate** - Data throughput analysis
7. **LDEV Response Time** - Latency analysis
8. **LDEV Read Hit** - Cache hit analysis for read-heavy workloads

### Configuration
To analyze a different dataset:
1. Update `ROOT_DIR` to point to your data directory
2. Update `DATA_FOLDER_NAME` to match your extracted folder name
3. Adjust thresholds in the Global Constants section as needed
4. Re-run all cells